In [ ]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")

import gdsfactory as gf
# TODO
# 1. Change one side of the cell_temp to be comparation with the width
# 2. Different with the width and same width with gap

cell_temp = gf.Component()

2025-12-15 14:46:21.616 | INFO     | kfactory.kcell:show:8777 - klive v0.3.3: Opened file '/Users/bubble/Desktop/Project/T_sensor/T_sensor/build/gds/1818199791.oas'


In [2]:
# cell
L = [1000, 2000]
g = 98
column_list = [3, 2]
w = 2
all_lines =[None] * len(L)
for i in range(len(L)):
    mesh = [None] * column_list[i]
    for j in range(column_list[i]):
        mesh[j] = gf.Component()    
    all_lines[i] = mesh
block = gf.Component()
origin = [[0,0], [0,0]]
x_pace = [1000,1000]
y_pace = [0, 3000+w]
for i in range(len(L)):
    for j in range(column_list[i]):
        # single grid
        row = (L[i]) // (g + w)
        sg = gf.components.rectangle(size=(g, g), layer=(8, 0))
        for m in range(row):
            for n in range(row):
                sf_ref = all_lines[i][j] << sg
                sf_ref.move((origin[i][0]+m*(g+w), origin[i][1]+n*(g+w)))
        # add pad
        pad = gf.components.rectangle(size=(g, g), layer=(10, 0))
        (all_lines[i][j] << pad).move((origin[i][0]-g/2+(L[i]-w)/2, origin[i][1]-g/2+(L[i]-w)/2))

        # frame of the grid
        frame = gf.components.rectangle(size=(L[i]+w, L[i]-w), layer=(9, 0))
        frame_ref = all_lines[i][j] << frame
        frame_ref.move((origin[i][0]-w, origin[i][1]))
        # frame_mid
        (block << all_lines[i][j]).move(((L[i]+x_pace[i])*j, y_pace[i]))
    # length mark
    T = gf.components.text(f"L={L[i]} gap={g}", size=50, layer=(1, 0))
    T_ref = all_lines[i][0] << T
    T_ref.move((2500, -150))
    # add to block
 
block.show()

In [3]:
# backside etching 5mm * 5mm (5743.44um)
backside = gf.Component()

size = [371.72*2+L[0]-w, 371.72*2+L[1]-w]
diff = (5743.44 - 5000) / 2
for i in range(4):
    backside_temp = gf.Component()
    backside_temp0 = gf.Component()
    backside_temp1 = gf.Component()
    for m in range(column_list[0]):
        backside_temp_1000 = gf.components.rectangle(size=(size[0], size[0]), layer=(3, 0))
        (backside_temp0 << backside_temp_1000).move((0, 0))
        (backside_temp0 << backside_temp_1000).move((2000, 0))
        (backside_temp0 << backside_temp_1000).move((4000, 0))
    for n in range(column_list[1]):
        backside_temp_2000 = gf.components.rectangle(size=(size[1], size[1]), layer=(3, 0))
        (backside_temp1 << backside_temp_2000).move((0, 0))
        (backside_temp1 << backside_temp_2000).move((3000, 0))
    backside_temp << backside_temp0
    (backside_temp << backside_temp1).move((0, 5743.44 - size[1]))
    backside_ref = backside << backside_temp
    if i == 0:
        backside_ref.move((-diff, -diff))
    elif i == 1:
        backside_ref.move((10000-diff, -diff))
    elif i == 2:
        backside_ref.move((-diff, 10000-diff))
    else:
        backside_ref.move((10000-diff, 10000-diff))
# backside.show()

In [4]:
# structure for each die
# structure for 4 sides
fblock = gf.Component()
for i in range(4):
    if i == 0:
        block_ref = fblock << block
    elif i == 1:
        block_ref = fblock << block
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block
        block_ref.move((0, 10000))

In [5]:
# order
order = gf.Component()
text_array = []
for i in range(4):
    text_array.append(gf.Component())

for i in range(4):
    # DML: double-clamped mesh large
    T = gf.components.text(f"WML {i+1}", size=20, layer=(1, 0))
    for j in range(4):
        order_ref = text_array[i] << T
        if j == 0:
            order_ref.move((-85, 0))
        elif j == 1:
            order_ref.move((-85, 5000))
        elif j == 2:
            order_ref.move((5020, 0))
        else:
            order_ref.move((5020, 5000))
    text_array_ref = order << text_array[i]
    if i == 0:
        pass
    elif i == 1:
        text_array_ref.move((10000, 0))
    elif i == 2:
        text_array_ref.move((0, 10000))
    else:
        text_array_ref.move((10000, 10000))





In [6]:
# backside frame 5mm * 5mm 
backframe = gf.Component()
for i in range(4):
    backframe_temp = gf.Component()
    backframe_temp0 = gf.Component()
    backframe_temp1 = gf.Component()
    for m in range(column_list[0]):
        backframe_temp_1000 = gf.components.rectangle(size=(L[0]-w, L[0]-w), layer=(3, 0))
        (backframe_temp0 << backframe_temp_1000).move((2000, 0))
        (backframe_temp0 << backframe_temp_1000).move((4000, 0))
    for n in range(column_list[1]):
        backframe_temp_2000 = gf.components.rectangle(size=(L[1]-w, L[1]-w), layer=(3, 0))
        (backframe_temp1 << backframe_temp_2000).move((0, 0))
        (backframe_temp1 << backframe_temp_2000).move((3000, 0))
    backframe_temp << backframe_temp0
    (backframe_temp << backframe_temp1).move((0, 5000-L[1]+w))
    backframe_ref = backframe << backframe_temp
    if i == 0:
        backframe_ref.move((0, 0))
    elif i == 1:
        backframe_ref.move((10000, 0))
    elif i == 2:
        backframe_ref.move((0, 10000))
    else:
        backframe_ref.move((10000, 10000))
# backside.show()

In [7]:
# boolean operation
# do backside etching - frame
outside = gf.boolean(A = backframe, B = fblock,  operation="not", layer1=(3, 0), layer2=(9, 0), layer=(2, 0))
outside = gf.boolean(A = outside, B = fblock,  operation="not", layer1=(2, 0), layer2=(10, 0), layer=(1, 0))

cell_temp << outside

# add holes to cell_temp
holes = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(8, 0), layer2=(10, 0), layer=(1, 0))
cell_temp << holes

cell_temp << backside
# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(10, 0), layer=(1, 0))
cell_temp << marker
cell_temp << order
# frame
frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_web_large_gap = gf.Component()
cell_ref = cell_web_large_gap << cell_temp
cell_ref.move((2500, -7500))
# cell_web_large_gap.show()
# cell_web_large_gap.write_gds("mesh.gds")
# cell_web_large_gap.plot()

Unnamed_75: ports [], vinsts=[] info=Info() kcl=KCLayout(name='DEFAULT', layout=<klayout.dbcore.Layout object at 0x11ad053d0>, layer_enclosures=LayerEnclosureModel(root={'a4b5fcfc': LayerEnclosure(layer_sections={}, main_layer=8/0, yaml_tag='!Enclosure'), 'c1ad96cb': LayerEnclosure(layer_sections={}, main_layer=10/0, yaml_tag='!Enclosure'), '4fb2b992': LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), '56396fc8': LayerEnclosure(layer_sections={}, main_layer=SLAB90 (3/0), yaml_tag='!Enclosure'), '6b4bcfc2': LayerEnclosure(layer_sections={}, main_layer=N (20/0), yaml_tag='!Enclosure'), 'a5f0d270': LayerEnclosure(layer_sections={}, main_layer=P (21/0), yaml_tag='!Enclosure')}), cross_sections={'a4b5fcfc_98000': SymmetricalCrossSection(width=98000, enclosure=LayerEnclosure(layer_sections={}, main_layer=8/0, yaml_tag='!Enclosure'), name='a4b5fcfc_98000'), 'c1ad96cb_98000': SymmetricalCrossSection(width=98000, enclosure=LayerEnclosure(layer_sections={}, main_layer=10/